# Lab 21: ARIMA & Prophet Forecasting — Guided Lab
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 57 min Core + 28 min Write it yourself + 10 min Extension + take-home Challenge

---

**Learning Objectives:**
- Pull CPI from FRED and prepare it for time series models: a monthly grid, a filled gap, and logs
- Use ACF and PACF plots to guess ARIMA orders before automatic selection
- Fit ARIMA and Prophet to U.S. inflation data and forecast 24 months ahead
- Check the residuals with the Ljung-Box test, and question a result that looks too clean
- Compare forecast accuracy with the Mean Absolute Scaled Error (MASE), and say what its denominator measures
- Write an expanding-window backtest that scores one-month-ahead ARIMA forecasts
- (Extension) Read Prophet's changepoints as automatic structural-break detection

**Dataset:** FRED CPI-U (CPIAUCSL): monthly, seasonally adjusted, from January 1990

**Prerequisites:** Lab 20 (your FRED key saved as a Colab secret; differencing; the ADF and KPSS tests) and Lab 15 (cross-validation).

**How this lab works:** Parts 1, 2, 4 and 5 are GUIDED: run each cell as it is and read the output. Parts 3 and 6 each have two blanks (`____`) for you to fill. "Write it yourself" is required: a short function you write from a blank cell. The Extension is optional, and the Challenge is a take-home.

**Plan on about 1¾ hours** (1 hour 40 to 1 hour 45 minutes) for the required work: setup, Parts 1-6, Write it yourself, and the README and submission. Each model fits in seconds; the check cell after Write it yourself refits ARIMA about 35 times, which takes several seconds.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI coding section: you write every line yourself. The one use of AI is drafting your README from the prompt at the end.

---

> ### What you write, and what you run
>
> **You write:**
> - **Four blanks (`____`)** in two cells: two in Step 3 (`seasonal` and `d` for `auto_arima`), and two in Step 6 (the slice positions behind the naive forecast's errors). Each is a word or a number.
> - **Write it yourself (required),** after Part 6: `expanding_mae()`, a function that refits ARIMA before each of the last months of a series, forecasts that month, and gives back the mean absolute error. About seven lines from a blank cell. The check cell below it must print "Passed"; it then prints your backtest's error beside the naive forecast's.
> - **Two written answers,** in text cells: one after Part 4, and one after Write it yourself.
>
> **You run and read:** everything else. Setup and Parts 1-6 are working code to learn from. Read each cell before you run it, and make sure you can say what each line does. The CHECK cell after Write it yourself is different: you only run it and read its message; you do not need to follow its code.
>
> **The Extension** (optional) runs as it is, with one question. **The Challenge** (take-home) has blanks of its own and a short written recommendation. The **README** is text only: the prompt at the end is the one place you use AI, and only for documentation.

## Setup

Colab does not come with `pmdarima` (the library behind `auto_arima` and `ARIMA`) or `fredapi`, so the
first line installs them, and upgrades `prophet` to 1.4 or later: Colab comes with prophet 1.1, whose
forecast and changepoints differ slightly from this notebook's expected output. pmdarima installs on
Python 3.12, the version Colab runs this term. The last two lines print your prophet and Python
versions; look at the Python one first if an import fails. If prophet shows a version below 1.4, use **Runtime → Restart session** and run this cell again.

Your FRED key comes from Colab's secret store, as in Lab 20; the comments in the cell say how to add it
if you have not.

In [ ]:
# GUIDED — run as-is
# Setup: install the libraries, import them, and connect to FRED
!pip install -q fredapi pmdarima "prophet>=1.4"

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')   # these libraries print convergence and deprecation warnings on most fits

from fredapi import Fred
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from pmdarima import auto_arima, ARIMA   # auto_arima searches for an order; ARIMA fits the order you give it
import prophet
from prophet import Prophet

# Your FRED key comes from Colab's secret store, never from this cell:
#   left sidebar -> key icon (Secrets) -> + Add new secret
#   Name: FRED_API_KEY    Value: your 32-character key    Notebook access: ON
# Free key: https://fredaccount.stlouisfed.org/apikeys
# You set this ONCE; every notebook on the same Google account then finds it.
FRED_API_KEY = ""
try:
    from google.colab import userdata                  # running in Colab
    FRED_API_KEY = userdata.get("FRED_API_KEY") or ""
except Exception:                                      # running locally
    import os
    FRED_API_KEY = os.environ.get("FRED_API_KEY", "")

if not FRED_API_KEY:
    # No secret set. getpass hides what you type, so the key stays out of this
    # notebook and out of its saved output either way.
    import getpass
    FRED_API_KEY = getpass.getpass("FRED API key (hidden as you type): ").strip()

if len(FRED_API_KEY) < 20:
    raise ValueError(
        "No FRED API key found. Get one free at "
        "https://fredaccount.stlouisfed.org/apikeys, then add it as a Colab "
        "secret named FRED_API_KEY with Notebook access switched ON -- missing "
        "that switch is the most common cause of this message."
    )

fred = Fred(api_key=FRED_API_KEY)

print(f'prophet {prophet.__version__} (this lab\'s Prophet numbers assume 1.4 or later)')
print(f'Libraries loaded (Python {sys.version.split()[0]}). Ready to pull FRED data.')

## Part 1: Pull CPI Data from FRED (GUIDED — 7 min)

CPIAUCSL, the Consumer Price Index for All Urban Consumers, is the most widely used measure of U.S.
inflation. We pull the seasonally adjusted version and take logs, because the first difference of log
CPI is approximately the monthly inflation rate:

$$\Delta \log(\text{CPI}_t) \approx \frac{\text{CPI}_t - \text{CPI}_{t-1}}{\text{CPI}_{t-1}}$$

The series stops at `END`, July 2026, so every number in this notebook stays the same all term; FRED
adds a month every few weeks.

**One month is missing:** BLS never published a CPI for October 2025. `.dropna()` hides the hole;
`.asfreq('MS')` puts the series on a grid of month starts, and the missing month comes back as NaN
(setting `cpi.index.freq = 'MS'` instead would raise an error, because the dates do not fit).
pmdarima refuses a series with NaN in it, so `.interpolate(method='time')` fills the gap on a straight
line between its neighbours. `', '.join(...)` glues the missing dates into one string.

In [ ]:
# GUIDED — run as-is
# Step 1: pull CPIAUCSL, put it on a monthly grid, fill the gap, and take logs
END = '2026-07-01'              # the last month used, here and in the Challenge
cpi = fred.get_series('CPIAUCSL', observation_start='1990-01-01', observation_end=END)
cpi = cpi.dropna()
cpi = cpi.asfreq('MS')          # one row per month start: a month FRED lacks becomes NaN

missing = cpi.index[cpi.isna()]
print('Missing months:', ', '.join(str(d.date()) for d in missing), '-- interpolating')
cpi = cpi.interpolate(method='time')
log_cpi = np.log(cpi)

print(f'Date range: {cpi.index[0].date()} to {cpi.index[-1].date()}')
print(f'Observations: {len(cpi)}')
print(f'Log CPI range: {log_cpi.min():.3f} to {log_cpi.max():.3f}')

**Expected output:** `Missing months: 2025-10-01 -- interpolating`, then a date range from 1990-01-01
to 2026-07-01, 439 observations, and log CPI from 4.848 to 5.811.

In [ ]:
# GUIDED — run as-is
# Step 1, continued: CPI and log CPI side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(cpi, color='navy', linewidth=0.8)
axes[0].set_title('U.S. CPI (CPIAUCSL)')
axes[0].set_ylabel('Index (1982-84 = 100)')
axes[1].plot(log_cpi, color='darkorange', linewidth=0.8)
axes[1].set_title('Log CPI')
axes[1].set_ylabel('log(CPI)')
plt.tight_layout()
plt.show()

## Part 2: ACF/PACF Identification (GUIDED — 8 min)

Before letting `auto_arima` choose orders automatically, we build
**visual intuition** with ACF and PACF plots of the differenced log CPI.

**Reading the plots:**
- **ACF** (Autocorrelation Function): If it cuts off after lag $q$, suggests MA($q$)
- **PACF** (Partial Autocorrelation Function): If it cuts off after lag $p$, suggests AR($p$)
- If both decay gradually, an ARMA($p$,$q$) mix may be needed

`log_cpi.diff()` subtracts each month's value from the next month's, leaving NaN in the first month,
and `.dropna()` drops it. `plot_acf(series, lags=24, ax=...)` draws the autocorrelations at lags 0 to
24 on the chart `ax` (lag 0 is always 1); the shaded band is the 95% band, and a bar inside it cannot be told apart from
zero. `method='ywm'` picks one of several ways to estimate the PACF.

Before you run Part 3, make your own guess of $p$ and $q$ from the two plots. Part 3 shows what
`auto_arima` picks.

In [ ]:
# GUIDED — run as-is
# Step 2: the differenced series, its ACF and its PACF
dlog_cpi = log_cpi.diff().dropna()      # approximately the monthly inflation rate

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(dlog_cpi, color='seagreen', linewidth=0.7)
axes[0].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[0].set_title('Δlog(CPI) ≈ Monthly Inflation')
plot_acf(dlog_cpi, lags=24, ax=axes[1])
axes[1].set_title('ACF of Δlog(CPI)')
plot_pacf(dlog_cpi, lags=24, ax=axes[2], method='ywm')
axes[2].set_title('PACF of Δlog(CPI)')
plt.tight_layout()
plt.show()

## Part 3: Fit ARIMA with auto_arima (YOUR TASK — 12 min)

Step 3 keeps the last 24 months aside as a test period: `log_cpi[:-24]` is every month except the
last 24, and `log_cpi[-24:]` is the last 24. On the rest, `auto_arima` (from pmdarima) searches over
$(p, d, q)$ orders and keeps the model with the lowest AIC. Fill in the two blanks:

- `seasonal` (`True` or `False`): whether the search should also look for a seasonal part. Part 1 says
  which version of CPI this is.
- `d`: how many times log CPI has to be differenced before it is stationary. Part 2 plotted log CPI
  differenced once: does that look stationary? Lab 20's Part 5 asked the same question of real GDP.

The other settings are given. `m=12` is the seasonal period of monthly data; it matters only when
`seasonal=True`. `stepwise=True` searches the orders with a fast heuristic rather than trying every
one, `trace=True` prints each model it tries with its AIC, and `suppress_warnings=True` hides warnings
from models that fail to fit. `arima_model.summary()` prints the chosen model; `.order` is its
$(p, d, q)$ and `.aic()` its AIC.

In [ ]:
# YOUR TASK — fill in the two blanks (____)
# Step 3: hold out the last 24 months, and let auto_arima choose an order on the rest
train = log_cpi[:-24]
test = log_cpi[-24:]

arima_model = auto_arima(
    train,
    seasonal=____,
    m=12,
    d=____,
    stepwise=True,
    trace=True,
    suppress_warnings=True,
)

print(arima_model.summary())
print(f'\nBest order: {arima_model.order}')
print(f'AIC: {arima_model.aic():.2f}')

**Expected output:** the search, one line per model tried, then the summary, `Best order: (0, 1, 2)`
and `AIC: -3821.21`. In the summary, `ma.L1` and `ma.L2` are the two MA coefficients, and `intercept`
(0.0022) is the average monthly change in log CPI: a drift of about 0.2% a month.

Step 3, continued forecasts the 24 test months with `.predict(n_periods=24)`, which gives back a Series
already labelled with the test months' dates, and prints the first forecast next to the actual value. `.iloc[0]` is the element at
position 0 of a Series, and `.date()` shows a timestamp as a plain date.

In [ ]:
# GUIDED — run as-is
# Step 3, continued: forecast the 24 test months, and plot them
arima_forecast = arima_model.predict(n_periods=24)
print(f'First test month, {test.index[0].date()}: forecast {arima_forecast.iloc[0]:.5f}, '
      f'actual {test.iloc[0]:.5f}')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(train[-60:], color='navy', linewidth=0.8, label='Train (last 5 yrs)')
ax.plot(test, color='navy', linewidth=0.8, linestyle='--', label='Actual')
ax.plot(arima_forecast, color='firebrick', linewidth=1.5, label='ARIMA Forecast')
ax.set_title(f'ARIMA{arima_model.order} Forecast — Log CPI')
ax.set_ylabel('log(CPI)')
ax.legend()
plt.tight_layout()
plt.show()

**Expected output:** `First test month, 2024-08-01: forecast 5.75035, actual 5.74959`, and a chart
in which the forecast runs as an almost straight line: after a month or two, the MA terms have nothing
left to add, and only the drift remains.

## Part 4: Residual Diagnostics (GUIDED — 10 min)

A well-specified ARIMA model should produce **white noise residuals** —
no remaining autocorrelation. The **Ljung-Box test** checks:

- $H_0$: Residuals are independently distributed (no autocorrelation)
- $H_1$: Residuals exhibit autocorrelation at some lag

If $p > 0.05$, we **fail to reject** — the model has captured the
autocorrelation structure adequately.

**Before you test, drop element 0.** The first element of `arima_model.resid()` is not a real
residual: with no month before January 1990 to forecast from, it is close to log CPI itself (about
4.85, against residuals of a few thousandths). Left in, it makes the standard deviation about a
hundred times too large, shrinks every autocorrelation toward zero, and Ljung-Box reports p = 1.0000
at every lag. Step 4 drops it with `[1:]` (every element from position 1 on) and prints the evidence.

In [ ]:
# GUIDED — run as-is
# Step 4: the residuals, without element 0
residuals = arima_model.resid()[1:]     # element 0 is close to the raw first value, not a residual: see above

print(f'element 0 of resid(): {arima_model.resid().iloc[0]:.5f}')
print(f'standard deviation: {np.std(arima_model.resid()):.4f} with element 0, '
      f'{np.std(residuals):.4f} without it')

**Expected output:** `element 0 of resid(): 4.84595`, and a standard deviation of `0.2376` with it
against `0.0024` without it.

Step 4, continued draws the residuals three ways: over time, their ACF (as in Part 2), and a histogram.
`density=True` scales the histogram's bars so that their total area is 1.

In [ ]:
# GUIDED — run as-is
# Step 4, continued: the residuals over time, their ACF, and their distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(residuals, color='firebrick', linewidth=0.7)
axes[0].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[0].set_title('ARIMA Residuals')
plot_acf(residuals, lags=24, ax=axes[1])
axes[1].set_title('ACF of Residuals')
axes[2].hist(residuals, bins=30, color='steelblue', edgecolor='white', density=True)
axes[2].set_title('Residual Distribution')
plt.tight_layout()
plt.show()

Now the test. `acorr_ljungbox(residuals, lags=[6, 12, 18, 24], return_df=True)` runs it once for each
lag in the list and gives back one row per lag: `lb_stat` is the statistic, `lb_pvalue` its p-value.
The test at lag h pools all the autocorrelations from lag 1 to lag h, so once the lag-12 test
rejects, the tests at 18 and 24 mostly inherit that rejection; they do not show three separate
problems. The last line is True if any p-value is below 0.05 (`.any()` asks whether at least one
entry of a True/False column is True).

In [ ]:
# GUIDED — run as-is
# Step 4, continued: the Ljung-Box test at lags 6, 12, 18 and 24
lb_results = acorr_ljungbox(residuals, lags=[6, 12, 18, 24], return_df=True)
print(lb_results)
print(f"\nAny significant autocorrelation remaining? {(lb_results['lb_pvalue'] < 0.05).any()}")

**Expected output:** four rows, with p-values of 0.5795 at lag 6, 0.0033 at lag 12, 0.0013 at lag 18
and 0.0022 at lag 24, then `Any significant autocorrelation remaining? True`. If every p-value reads
1.0, element 0 is back in your residuals: rerun Step 4 from its first cell.

### Interpretation Question

1. The lag-6 test passes and the lag-12 test fails, so the problem sits somewhere in lags 7 to 12. Which lags hold it? Find them in the residual ACF, the middle chart of Step 4, continued (bars outside the shaded band), and say what you would try next in Step 3, given that Box-Jenkins is a loop (identify, estimate, diagnose, then back to identify).

*Your answer here:*

1.

## Part 5: Fit Prophet (GUIDED — 8 min)

Prophet, released by Facebook (now Meta) in 2017, is a decomposition-based forecasting model: it
writes the series as a trend plus seasonal terms plus holiday effects. Its trend is a sequence of
straight lines, and it detects the dates where the slope changes, its **changepoints** (structural
breaks in trend), by itself.

**Key parameters:**
- `changepoint_prior_scale`: Controls flexibility of trend changes (lower = smoother); 0.05 is the default
- `yearly_seasonality`: Whether to fit annual seasonal patterns

Prophet requires a DataFrame with columns `ds` (datestamp) and `y` (value).
`pd.DataFrame({'ds': ..., 'y': ...})` builds one from a dictionary: each key becomes a column name.
`make_future_dataframe(periods=24, freq='MS')` adds the 24 month starts after the training months, and
`.predict()` gives back a DataFrame with one row per date: `yhat` is the forecast, and `yhat_lower` and
`yhat_upper` are the edges of its 80% interval. `.set_index('ds').loc[test.index]` keeps the rows of
the 24 test months. Prophet prints a few log lines from cmdstanpy, the program that fits it.

In [ ]:
# GUIDED — run as-is
# Step 5: fit Prophet on the same training months, and forecast the same 24 test months
train_df = pd.DataFrame({'ds': train.index, 'y': train.values})

prophet_model = Prophet(
    changepoint_prior_scale=0.05,
    yearly_seasonality=True,     # an annual pattern, although CPIAUCSL is seasonally adjusted
    weekly_seasonality=False,    # monthly data has no weekly or daily pattern
    daily_seasonality=False,
)
prophet_model.fit(train_df)

future = prophet_model.make_future_dataframe(periods=24, freq='MS')
prophet_pred = prophet_model.predict(future)
prophet_test = prophet_pred.set_index('ds').loc[test.index]   # the rows of the 24 test months
prophet_forecast = prophet_test['yhat']

Step 5, continued puts both forecasts on one chart. `fill_between(x, lower, upper)` shades the area
between two lines, here Prophet's 80% interval; `alpha=0.15` makes the shading mostly transparent.

In [ ]:
# GUIDED — run as-is
# Step 5, continued: both forecasts against the actual values
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(train[-60:], color='navy', linewidth=0.8, label='Train (last 5 yrs)')
ax.plot(test, color='navy', linewidth=0.8, linestyle='--', label='Actual')
ax.plot(arima_forecast, color='firebrick', linewidth=1.5, label='ARIMA Forecast')
ax.plot(prophet_forecast, color='steelblue', linewidth=1.5, label='Prophet Forecast')
ax.fill_between(test.index, prophet_test['yhat_lower'], prophet_test['yhat_upper'],
                alpha=0.15, color='steelblue', label='Prophet 80% interval')
ax.set_title('ARIMA vs Prophet — Log CPI Forecast')
ax.set_ylabel('log(CPI)')
ax.legend()
plt.tight_layout()
plt.show()

## Part 6: Compare Forecasts with MASE (YOUR TASK — 12 min)

The **Mean Absolute Scaled Error (MASE)** compares a model's forecast
errors against a naive (random walk) baseline:

$$\text{MASE} = \frac{\frac{1}{h} \sum_{t=1}^{h} |y_t - \hat{y}_t|}{\frac{1}{T-1} \sum_{t=2}^{T} |y_t - y_{t-1}|}$$

- **MASE < 1**: the forecast's errors are smaller than the one-step naive error on the training months
- **MASE = 1**: the same size
- **MASE > 1**: larger than the one-step naive error on the training months

**What this evaluation actually does.** Both models were fitted **once**, on
everything up to the last 24 months, and both forecasts run 24 steps ahead
from that single origin. The denominator is the one-step naive error measured
on the **training** data — the standard MASE scaling from Hyndman & Koehler
(2006), which is what makes the number comparable across series of different
units.

Note the asymmetry that creates: a 24-step-ahead forecast is a far harder
problem than the one-step-ahead naive errors it is being divided by. A MASE
above 1 here does **not** mean the model loses to a random walk at the same
horizon.

The fairer comparison is an **expanding window** — refit at every origin,
forecast one step, then expand — which puts numerator and denominator on the
same horizon. Done in full, it means a new `auto_arima` search and a new
Prophet fit at each of the 24 origins, and Part 6 does not run it. "Write it
yourself" runs the ARIMA half, with Step 3's order held fixed, which needs only
one quick fit per month.

**The blanks.** In `compute_mase`, the naive forecast of each training month is the month before it,
so the denominator lines the training values up against themselves, shifted by one month: fill in the
two slice positions so that each value $y_t$ meets $y_{t-1}$. `.values` is the plain NumPy array of a
Series; `a[i:]` keeps the elements from position `i` on, `a[:j]` those before position `j`, and a
negative position counts from the end. The last line recovers the denominator from the printed
numbers: MASE is MAE divided by the denominator, so the denominator is MAE divided by MASE.

In [ ]:
# YOUR TASK — fill in the two blanks (____)
# Step 6: MAE and MASE for ARIMA and Prophet
def compute_mase(actual, forecast, train_series):
    '''MAE of the forecast, divided by the MAE of the one-month naive forecast on train_series.'''
    mae_forecast = np.mean(np.abs(actual - forecast))
    naive_errors = np.abs(train_series.values[____:] - train_series.values[:____])   # |y_t - y_{t-1}|
    return mae_forecast / np.mean(naive_errors)

mase_arima = compute_mase(test.values, arima_forecast.values, train)
mase_prophet = compute_mase(test.values, prophet_forecast.values, train)
mae_arima = np.mean(np.abs(test.values - arima_forecast.values))
mae_prophet = np.mean(np.abs(test.values - prophet_forecast.values))

print(f'ARIMA   MAE {mae_arima:.5f}   MASE {mase_arima:.4f}')
print(f'Prophet MAE {mae_prophet:.5f}   MASE {mase_prophet:.4f}')
print(f'denominator, the naive one-month MAE on the training months: {mae_arima / mase_arima:.6f}')

**Expected output:** ARIMA MAE `0.00344`, MASE `1.2680`; Prophet MAE `0.00533`, MASE `1.9659`; and a
denominator of `0.002713`. Both MASE values are above 1, and ARIMA's is the lower. Three slips print a
different ARIMA MASE: `0.6643` means the slices are two months apart (`[2:]` and `[:-2]`), which would
put ARIMA below 1; `0.1313` means twelve months apart (a seasonal naive); `0.0076` means the second slice is `[:1]`,
so every month is compared with January 1990. Before you read
anything into "above 1", reread "What this evaluation actually does" above; "Write it yourself" then
measures the comparison it describes.

---

## Write it yourself (required — 28 min)

This part is required. The check cell below it must print "Passed" before you submit.

**The task.** Part 6 said the fair comparison is an expanding window. Run it for ARIMA, one month
ahead. Write `expanding_mae(series, order, n_test)` that, for each of the last `n_test` months of
`series`:

1. fits `ARIMA(order=order)` on every month before that month, and on nothing after it;
2. forecasts one month ahead, which is that month;
3. records the absolute error of that forecast,

and gives back the mean of the `n_test` absolute errors, as one number. Once the check passes, it runs
`expanding_mae(log_cpi, arima_model.order, 24)`: Step 3's model, refitted before each of the 24 test
months with only the months before it.

- It takes about seven lines: the `def` line, an empty list, a `for` line, the fit, the forecast, the
  append, and the `return`. Lab 15's `kfold_mse()` had the same shape.
- `ARIMA` was imported in Setup, from pmdarima: it is the model `auto_arima` searched over, with the
  order fixed instead of searched. `ARIMA(order=order).fit(y)` fits it to the series `y`, and
  `.predict(n_periods=1)` then forecasts the month after the end of `y`, as Step 3's
  `.predict(n_periods=24)` forecast 24. The forecast comes back as a Series of one value: `.iloc[0]`
  is that value.
- `series.iloc[k]` is the month at position `k`, and `series[:k]` is every month before it (positions
  0 to k − 1). `len(series)` is the number of months.
- Give back the mean absolute error itself: not squared, not its square root, not times 100, not
  rounded.
- To test it yourself: `expanding_mae(log_cpi[:len(train) + 1], arima_model.order, 1)` makes Step 3's
  first forecast again, so it should give the gap between the two numbers Step 3 printed for the first
  test month, about 0.00076. Python may display it as `np.float64(0.00075...)`: that is just a number.

Then run the check cell. It compares your function with Step 3's first forecast and with the first four
test months, and tries it with another order and other series. It refits ARIMA about 35 times, which
takes several seconds. `assert` stops with its message when its condition is False; the message tells you
what to fix.

In [ ]:
# WRITE IT YOURSELF (required): expanding_mae(series, order, n_test)

In [ ]:
# CHECK — for "Write it yourself" above

# globals() holds every name defined so far in this notebook
if "expanding_mae" not in globals():
    print("Not written yet: write expanding_mae() in the cell above and run it, then run this check.")
elif not all(name in globals() for name in ["arima_model", "arima_forecast", "mae_arima"]):
    print("Run Parts 1-6 first, Steps 3 and 6 included, then this check.")
else:
    # Errors recorded for this lab's data (to END = 2026-07-01) and Step 3's ARIMA(0, 1, 2):
    # the first four test months done right, and what some common slips give for the first one.
    RIGHT_4 = [0.0007580058486578167, 0.00031649461388383315, 0.000589125869422169, 0.00031143633371311097]
    UPDATED_4 = 0.0004931906345748693
    SLIPS_1 = {
        "future": 0.0018145697841971398,
        "early": 0.0006200819514283751,
        "gap": 0.0021910690486039996,
        "in-sample": 0.0007583128464467848,
        "no drift": 0.0004318917741867878,
        "trend t": 0.000765436887556703,
    }

    step3_order = arima_model.order
    origin = len(train)                         # the position of the first test month in log_cpi
    same_data = (str(log_cpi.index[-1].date()) == "2026-07-01" and len(test) == 24
                 and len(log_cpi) == origin + 24 and np.allclose(log_cpi.values[:origin], train.values))
    assert same_data, ("log_cpi, train or test is not what Parts 1-3 build (data to END = 2026-07-01, the "
                       "last 24 months held out). Write only the function, rerun Parts 1-6, then this cell.")
    assert step3_order == (0, 1, 2) and arima_model.seasonal_order[:3] == (0, 0, 0) and arima_model.with_intercept, (
        "Step 3's model is not the one its expected output describes. Compare Step 3's output with its "
        "expected output, fix its blanks, rerun Parts 3-6, then this cell.")

    def call(series, order, n_test):
        """Run your expanding_mae; if it goes wrong, say where to look."""
        # try/except catches an error inside your function, so the message can name the cause
        try:
            out = expanding_mae(series, order, n_test)
        except Exception as e:
            msg = str(e)
            hint = "Read the error above: it says what went wrong."
            if "positional argument" in msg or "unexpected keyword" in msg or "tuple" in msg:
                hint = "expanding_mae takes three arguments, in this order: series, order, n_test."
            elif "'numpy.ndarray' object has no attribute 'iloc'" in msg:
                hint = ("predict() gave back a plain array, without dates. That happens when the model was "
                        "fitted on .values, or after .update(). Fit a new ARIMA(order=order) on a slice of "
                        "series before each month.")
            elif isinstance(e, KeyError):
                hint = ("On a Series with dates, [k] or [0] looks k up as a date, not a position. Use "
                        ".iloc[k] for the month at position k, and .iloc[0] for the forecast's value.")
            elif isinstance(e, IndexError):
                hint = ("It reached for a position series does not have. The last month of series is at "
                        "position len(series) - 1.")
            # msg[:200] keeps the first 200 characters of the error message
            raise AssertionError(f"expanding_mae on a series of {len(series)} months, order {order}, "
                                 f"n_test={n_test}, stopped with an error:\n  {msg[:200]}\n{hint}") from None
        assert out is not None, "expanding_mae gave back nothing. Does it end with a return line?"
        assert np.size(out) == 1 or np.size(out) != n_test, (
            f"expanding_mae gives back {n_test} values, one error per month. Give back their mean.")
        assert np.size(out) == 1, (
            f"expanding_mae gives back {np.size(out)} values. Give back one number: the mean absolute error.")
        value = float(np.ravel(out)[0])
        assert not np.isnan(value), (
            "expanding_mae gives nan, the mean of an empty list: the loop ran zero times, or the mean was "
            "taken over an empty part of the list. Check where the range starts: it should cover the last "
            "n_test positions of series.")
        return value

    def near(a, b, rtol=1e-5):
        return np.isclose(a, b, rtol=rtol, atol=0)

    # Step 3's first forecast is a one-month-ahead forecast made from the end of train
    s1 = log_cpi[:origin + 1]                       # the training months, then the first test month
    gap = test.iloc[0] - arima_forecast.iloc[0]     # Step 3's error in that month
    want = abs(gap)
    got = call(s1, step3_order, 1)

    if not near(got, want, 1e-7):
        assert not near(got, want ** 2), "Your error is squared. Give back the mean absolute error: no squares."
        assert not near(got, 100 * want), (
            "Your answer is 100 times the right one: a percent. Give back the error in log points, as "
            "Step 6 does.")
        assert not (near(got, gap) or near(got, -gap)), (
            "Your error keeps its sign, so errors above and below cancel. Take the absolute value of each.")
        assert not (near(got, want / test.iloc[0]) or near(got, 100 * want / test.iloc[0])), (
            "Your error is divided by the actual value: a percentage error (MAPE). Give back the absolute "
            "error itself.")
        assert not near(got, abs(np.exp(test.iloc[0]) - np.exp(arima_forecast.iloc[0]))), (
            "Your error is in CPI index points: you undid the log with exp(). Give back the error in log "
            "points, on the series as it is given.")
        # round(v, places) keeps that many decimals; try 1 to 8 of them
        for places in [1, 2, 3, 4, 5, 6, 7, 8]:
            assert not np.isclose(got, round(want, places), rtol=0, atol=1e-12), (
                "Your number is rounded. Give back the full number; round only when you print.")
        assert not near(got, abs(test.iloc[0] - train.iloc[-1])), (
            "That is the naive forecast's error: the month before compared with the month itself. Compare "
            "the month with the ARIMA forecast instead.")
        assert not near(got, call(s1, step3_order, 2), 1e-9), (
            "Asked for 1 month and for 2 months, expanding_mae gives the same answer. Does it use its "
            "n_test argument?")
        assert not near(got, call(log_cpi, step3_order, 1), 1e-9), (
            "Given the first months of log_cpi, expanding_mae answers as if it had all of log_cpi. Does it "
            "use its series argument everywhere, or the notebook's log_cpi?")
        assert not near(got, abs(train.iloc[-1] - arima_forecast.iloc[0])), (
            "Your error compares the forecast with the month before the one it forecasts. Compare it with "
            "the month the model has not seen.")
        assert not near(got, SLIPS_1["early"]), (
            "Your function forecasts one month too early: the last month of series is never forecast. The "
            "months to forecast are the last n_test months of series.")
        assert not near(got, SLIPS_1["gap"]), (
            "Your fit leaves out the month just before the one it forecasts, so the forecast is for the "
            "wrong month. Fit on every month before the one you forecast.")
        assert not near(got, SLIPS_1["future"]), (
            "Your fit includes the month it then forecasts: the future leaks into the model. Fit on the "
            "months before it only.")
        assert not near(got, SLIPS_1["in-sample"]), (
            "Your function fits once, on the whole series, and reads off its in-sample predictions: the "
            "month being forecast helped fit the model. Refit before each month, on the months before it.")
        assert not near(got, SLIPS_1["no drift"]), (
            "Your model has no intercept (the drift): with_intercept=False, or statsmodels' ARIMA, which "
            "leaves it out. Use pmdarima's ARIMA(order=order), imported in Setup, as it is.")
        assert not near(got, SLIPS_1["trend t"]), (
            "Close, but this is statsmodels' ARIMA with trend='t', a slightly different model. Use "
            "pmdarima's ARIMA(order=order), imported in Setup, which Step 3's model is.")
        assert False, (
            f"expanding_mae(log_cpi[:len(train) + 1], arima_model.order, 1) gives {got:.6f}. That call "
            "makes Step 3's first forecast again, so it should give the gap between the forecast and the "
            f"actual value Step 3 printed for {test.index[0].date()}: {want:.6f}. Check three things: the "
            "model (pmdarima's ARIMA, imported in Setup, with order=order), the months it is fitted on, "
            "and the month its forecast is compared with.")

    assert not near(call(s1, (0, 1, 1), 1), got, 1e-9), (
        "With order (0, 1, 1), expanding_mae gives the same answer as with Step 3's order. Does it fit "
        "ARIMA(order=order), with its own order argument? Step 3's arima_model, or auto_arima, would "
        "ignore it.")
    # -s1 is log CPI upside down: every forecast error changes sign, but not its size
    assert call(-s1, step3_order, 1) >= 0, (
        "Turned upside down, the series gives a negative answer, so your errors keep their sign and can "
        "cancel. Take the absolute value of each error.")
    assert not near(call(cpi[:origin + 1], step3_order, 1), got, 1e-9), (
        "Given CPI itself rather than log CPI, expanding_mae gives the same answer. Does it use its "
        "series argument everywhere, or the notebook's log_cpi?")

    # the first 4 test months: one at a time, then together
    singles = [got] + [call(log_cpi[:origin + j + 1], step3_order, 1) for j in [1, 2, 3]]
    for j in [1, 2, 3]:
        if not near(singles[j], RIGHT_4[j], 1e-4):
            assert not near(singles[j], abs(test.iloc[j] - arima_forecast.iloc[0])), (
                f"For test month {j + 1}, your forecast is made from the end of train, as Step 3's was. "
                "Fit on every month before the one you forecast, so the window grows.")
            assert False, (
                f"For test month {j + 1} alone (n_test=1 on log_cpi[:len(train) + {j + 1}]), expanding_mae "
                f"gives {singles[j]:.6f}; it should be {RIGHT_4[j]:.6f}. Check that each fit uses all the "
                "months before the one it forecasts, and nothing else.")
    got4 = call(log_cpi[:origin + 4], step3_order, 4)
    if not near(got4, np.mean(singles), 1e-9):
        assert not near(got4, np.sum(singles)), (
            "For 4 months, your answer is the sum of the four errors. Give back their mean.")
        assert not near(got4, np.sqrt(np.mean(np.square(singles)))), (
            "For 4 months, your answer is the square root of the mean squared error (an RMSE). Give back "
            "the mean absolute error.")
        assert not near(got4, np.mean(np.square(singles))), (
            "For 4 months, your answer is the mean squared error. Give back the mean absolute error.")
        assert not near(got4, np.median(singles)), (
            "For 4 months, your answer is the median of the errors. Give back their mean.")
        assert not near(got4, np.mean(np.abs(test.values[:4] - arima_forecast.values[:4]))), (
            "For 4 months, you fit once, before the first of them, and forecast all four from there: "
            "Step 6's single origin. Refit before every month, on all the months before it.")
        assert not near(got4, np.mean(np.abs(test.values[:4] - arima_forecast.iloc[0]))), (
            "Every month gets Step 3's first forecast. Fit a new model for each month, on the months "
            "before it, instead of reusing arima_model or fitting on train.")
        assert not near(got4, UPDATED_4, 2e-4), (
            "Your model is fitted once and then updated with each new month, so its coefficients never "
            "change. Fit a new ARIMA(order=order) before every month.")
        assert not near(got4, np.mean(singles[1:])), (
            "For 4 months, your answer leaves out the first month: the loop starts one month late. It "
            "should cover the last n_test positions of series.")
        assert not near(got4, singles[-1]), (
            "For 4 months, your answer is the last month's error alone: each pass through the loop throws "
            "away the months before it. Make the empty list before the loop, and append to it.")
        assert not near(got4, singles[0]), (
            "For 4 months, your answer is the first month's error alone. Is the return line inside the "
            "for loop? Move it out, so that every month runs first.")
        assert not near(got4, call(log_cpi[:origin + 4], step3_order, 3), 1e-9), (
            "For 3 months and for 4, expanding_mae gives the same answer. Does it use its n_test argument?")
        assert False, (
            f"For the first 4 test months, expanding_mae gives {got4:.6f}, but the mean of its four "
            f"one-month answers is {np.mean(singles):.6f}. Check which months the loop runs over (the "
            "last n_test months of series) and that each fit uses only the months before the one it "
            "forecasts.")

    yours = call(log_cpi, step3_order, 24)
    naive = log_cpi.diff().abs().iloc[-24:].mean()   # each test month forecast by the month before it
    print("Passed. expanding_mae repeats Step 3's first forecast and the next three months, averages them, "
          "and uses the series, order and number of months it is given.")
    print("\nOne-month-ahead forecasts of the last 24 months, each made from the months before it:")
    rows = [(f"ARIMA{step3_order}, refitted every month", f"MAE {yours:.5f}"),
            ("naive (last month's value)", f"MAE {naive:.5f}"),
            ("ARIMA MAE / naive MAE", f"{yours / naive:.3f}")]
    for label, value in rows:
        print(f"  {label:<40}{value}")
    print(f"\nStep 6, one fit and 1 to 24 months ahead: ARIMA MAE {mae_arima:.5f}, MASE {mase_arima:.4f}")

### Interpretation Question

1. Your check printed two errors for the same 24 months, both one month ahead: ARIMA refitted before every month, and the naive forecast (last month's value). Does ARIMA beat the naive forecast once both forecast the same distance ahead? Quote both numbers.

*Your answer here:*

1.

---

## Extension (Optional — 10 min): Visualize Prophet Changepoints

One of Prophet's most useful features is automatic **changepoint detection**.
Changepoints are dates where the trend slope shifts — analogous to the
structural breaks we detected with PELT in Lab 20, but estimated jointly
with the forecast model.

Prophet places 25 evenly spaced candidate dates and shrinks their slope changes (`delta`) toward
zero, as LASSO shrinks coefficients, so that only a few stay large. Step 7 draws the forecast with the
changepoints whose `delta` is at least 0.01 (9 of the 25 fall below that, so the chart draws 16
lines), prints the range of the candidate dates, and lists the five largest. `prophet_model.params['delta']` holds the 25
slope changes, and `.flatten()` makes them one flat array. `nlargest(5, 'abs_delta')` keeps the five
rows with the largest absolute change, and `for _, row in df.iterrows()` walks through rows one at a
time (`_` is the row label, which is not needed).

**Your answer.** Compare the five dates with events you know: the Great Recession, COVID, the
2021-22 inflation surge. Which of them appear, and which could Prophet not have placed a changepoint
on? Why not?

In [ ]:
# GUIDED — run as-is (optional)
# Step 7: the changepoints Prophet found, on its forecast chart and as a list
from prophet.plot import add_changepoints_to_plot

fig = prophet_model.plot(prophet_pred)
ax = fig.gca()
add_changepoints_to_plot(ax, prophet_model, prophet_pred)
ax.set_title('Prophet Forecast with Detected Changepoints')
ax.set_ylabel('log(CPI)')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

cp_df = pd.DataFrame({
    'date': prophet_model.changepoints,
    'delta': prophet_model.params['delta'].flatten(),
})
cp_df['abs_delta'] = np.abs(cp_df['delta'])
print(f"{len(cp_df)} candidate changepoints, {cp_df['date'].min().date()} to {cp_df['date'].max().date()}")
print('Top 5 by magnitude:')
for _, row in cp_df.nlargest(5, 'abs_delta').iterrows():
    direction = 'acceleration' if row['delta'] > 0 else 'deceleration'
    print(f"  {row['date'].date()} — delta = {row['delta']:.6f} ({direction})")

*Your answer here:*

---

## Challenge (Take-Home): Apply to a Different Series

Apply the full ARIMA + Prophet pipeline to the **U.S. Unemployment Rate**
(UNRATE). This series behaves very differently from CPI — it is
mean-reverting, spiky during recessions, and does not have a persistent
upward trend. Fill in the blanks to adapt the pipeline, then, in a text cell below it, say which model
you would recommend for UNRATE and why, quoting your two MASE values.

If you skip the Challenge, leave its cell unrun, so your submitted notebook shows no error.

In [ ]:
# YOUR TASK — take-home Challenge: fill in the blanks (____)

# Step A: pull UNRATE and put it on the same monthly grid; it has the same October 2025 gap
unrate = fred.get_series('UNRATE', observation_start='1990-01-01', observation_end=END)
unrate = unrate.dropna()
unrate = unrate.asfreq('MS').interpolate(method='time')
unrate.name = 'UNRATE'

# Step B: no log here: unemployment is a percent between about 3 and 15, and its swings
# do not grow with its level, as CPI's do
y = unrate

# Step C: hold out the last 24 months
train_u = y[:-24]
test_u = y[-24:]

# Step D: auto_arima (UNRATE is a seasonally adjusted series)
arima_u = auto_arima(
    train_u,
    seasonal=____,
    m=____,              # the seasonal period of monthly data
    d=____,
    stepwise=True,
    trace=True,
    suppress_warnings=True,
)
print(f'Best ARIMA order: {arima_u.order}')

# Step E: Prophet
train_u_df = pd.DataFrame({'ds': train_u.index, 'y': train_u.values})
prophet_u = Prophet(
    changepoint_prior_scale=____,   # how freely the trend may bend: Part 5 used the default
    yearly_seasonality=____,
)
prophet_u.fit(train_u_df)

# Step F: forecasts and MASE
arima_fc_u = arima_u.predict(n_periods=24)
future_u = prophet_u.make_future_dataframe(periods=24, freq='MS')
prophet_pred_u = prophet_u.predict(future_u)
prophet_fc_u = prophet_pred_u.set_index('ds').loc[test_u.index, 'yhat']

mase_arima_u = compute_mase(test_u.values, arima_fc_u.values, train_u)
mase_prophet_u = compute_mase(test_u.values, prophet_fc_u.values, train_u)
print(f'\nUNRATE MASE — ARIMA: {mase_arima_u:.4f}, Prophet: {mase_prophet_u:.4f}')

# Step G: plot both forecasts
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(train_u[-60:], color='navy', linewidth=0.8, label='Train')
ax.plot(test_u, color='navy', linewidth=0.8, linestyle='--', label='Actual')
ax.plot(arima_fc_u, color='firebrick', linewidth=1.5, label=f'ARIMA (MASE={mase_arima_u:.3f})')
ax.plot(prophet_fc_u, color='steelblue', linewidth=1.5, label=f'Prophet (MASE={mase_prophet_u:.3f})')
ax.set_title('ARIMA vs Prophet — U.S. Unemployment Rate')
ax.set_ylabel('Percent')
ax.legend()
plt.tight_layout()
plt.show()

*Your recommendation here:*

---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Pulled monthly U.S. CPI (CPIAUCSL) from FRED, January 1990 to
  [YOUR VALUE], filled the one missing month (October 2025), and
  took logs
* Held out the last 24 months and let auto_arima choose an order on
  the rest: ARIMA[YOUR VALUE], AIC [YOUR VALUE]
* Tested the residuals with Ljung-Box at lags 6, 12, 18 and 24:
  p-values [YOUR VALUE]
* Fitted Prophet on the same months, and scored both 24-month
  forecasts: MASE [YOUR VALUE] for ARIMA and [YOUR VALUE] for Prophet
* Wrote a function, expanding_mae(), that refits ARIMA before each of
  the 24 test months and forecasts one month ahead: MAE [YOUR VALUE],
  against [YOUR VALUE] for the naive forecast on the same months
* (Only if I did the Challenge) On the unemployment rate: [YOUR RESULT]

**Please write a README.md entry including:**
1. Project Title: Forecasting U.S. Inflation with ARIMA and Prophet
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust', 'comprehensive'
or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab21-arima-prophet`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab21-arima-prophet`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab21-arima-prophet`, branch `main`,
   commit message `Lab 21 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.